# Pandas & Matplotlib
This part of the project uses Python (pandas and Matplotlib) to take the UK train rides data from raw CSV to analysis-ready tables.
- **Cleaning**
- **ETL** 
- **Analysis**

## Cleaning


In [29]:
#hab2a azabt el documentation bta3o ba3dain

## ETL
The purpose of this ETL process is to create the fact & dimentions tables.
<table style="margin-left:0">
<tr><th>Step</th><th>Purpose</th></tr>
<tr><td>Extract</td><td>read the cleaned CSV</td></tr>
<tr><td>Transform</td><td>split it into dimensions and a fact table</td></tr>
<tr><td>Load</td><td>put the results into the warehouse</td></tr>
</table>

Following the star scehma:
<img src="../docs/star_schema.png" alt="Star schema" width="875">

### dim_stations

In [30]:
import pandas as pd
df = pd.read_csv('../data/cleaned/railway.csv', keep_default_na=False, na_values=[''])
df.columns = df.columns.str.lower().str.replace(" ", "_")
dim_stations = pd.DataFrame(sorted(pd.concat([df["departure_station"], df["arrival_destination"]]).unique()), columns=["station_name"])
dim_stations["station_id"] = range(1, len(dim_stations) + 1)
dim_stations = dim_stations[["station_id", "station_name"]]
dim_stations.head(3)

,station_id,station_name
0,1,Birmingham New Street
1,2,Bristol Temple Meads
2,3,Cardiff Central


### dim_railcard

In [31]:
dim_railcard = pd.DataFrame(sorted(df["railcard"].unique()), columns=["railcard_type"])
dim_railcard["railcard_id"] = range(1, len(dim_railcard) + 1)
dim_railcard = dim_railcard[["railcard_id", "railcard_type"]]
dim_railcard

,railcard_id,railcard_type
0,1,Adult
1,2,Disabled
2,3,None
3,4,Senior


### dim_sale_details

In [32]:
dim_sale_details = df[["purchase_type", "payment_method", "ticket_class", "ticket_type"]].drop_duplicates().sort_values(["purchase_type", "payment_method", "ticket_class", "ticket_type"]).reset_index(drop=True)
dim_sale_details["sale_details_id"] = range(1, len(dim_sale_details) + 1)
dim_sale_details = dim_sale_details[["sale_details_id", "purchase_type", "payment_method", "ticket_class", "ticket_type"]]
dim_sale_details.head(3)

,sale_details_id,purchase_type,payment_method,ticket_class,ticket_type
0,1,Online,Contactless,First Class,Advance
1,2,Online,Contactless,First Class,Anytime
2,3,Online,Contactless,First Class,Off-Peak


### dim_journey

In [33]:
df = df.rename(columns={"date_of_journey": "journey_date", "arrival_time": "scheduled_arrival_time"})

journey_lookup = (
    df[["journey_date", "departure_time", "scheduled_arrival_time", "actual_arrival_time", "journey_status", "reason_for_delay", "departure_station", "arrival_destination"]]
    .drop_duplicates()
    .sort_values(["journey_date", "departure_time", "departure_station", "arrival_destination"])
    .reset_index(drop=True)
)

journey_lookup["journey_id"] = range(1, len(journey_lookup) + 1)

dim_journey = journey_lookup[["journey_id", "journey_date", "departure_time", "scheduled_arrival_time", "actual_arrival_time", "journey_status", "reason_for_delay"]]
dim_journey.head(3)

,journey_id,journey_date,departure_time,scheduled_arrival_time,actual_arrival_time,journey_status,reason_for_delay
0,1,2024-01-01,00:00:00,00:30:00,00:30:00,On Time,No Delay
1,2,2024-01-01,00:00:00,01:50:00,02:07:00,Delayed,Signal Failure
2,3,2024-01-01,01:30:00,02:30:00,02:30:00,On Time,No Delay


### fact_ticket

In [39]:
fact_ticket = (
    df.merge(dim_railcard.rename(columns={"railcard_type": "railcard"}), on="railcard", how="left", validate="m:1")
    .merge(dim_stations.rename(columns={"station_id": "departure_station_id", "station_name": "departure_station"}), on="departure_station", how="left", validate="m:1")
    .merge(dim_stations.rename(columns={"station_id": "arrival_station_id", "station_name": "arrival_destination"}), on="arrival_destination", how="left", validate="m:1")
    .merge(dim_sale_details, on=["purchase_type", "payment_method", "ticket_class", "ticket_type"], how="left", validate="m:1")
    .merge(journey_lookup[["journey_date", "departure_time", "departure_station", "arrival_destination", "journey_id"]], on=["journey_date", "departure_time", "departure_station", "arrival_destination"], how="left", validate="m:1")
    .rename(columns={"date_of_purchase": "purchase_date", "time_of_purchase": "purchase_time"})
    [["transaction_id", "journey_id", "railcard_id", "departure_station_id", "arrival_station_id", "sale_details_id", "purchase_date", "purchase_time", "price", "refund_request"]]
)

fact_ticket.head(3)

# fact_ticket.to_csv("../data/star/fact_ticket.csv", index=False)
# dim_railcard.to_csv("../data/star/dim_railcard.csv", index=False)
# dim_stations.to_csv("../data/star/dim_stations.csv", index=False)
# dim_sale_details.to_csv("../data/star/dim_sale_details.csv", index=False)
# dim_journey.to_csv("../data/star/dim_journey.csv", index=False)

,transaction_id,journey_id,railcard_id,departure_station_id,arrival_station_id,sale_details_id,purchase_date,purchase_time,price,refund_request
0,da8a6ba8-b3dc-4677-b176,34,1,15,12,4,2023-12-08,12:41:11,43,No
1,b0cdd1b0-f214-4197-be53,32,1,14,31,28,2023-12-16,11:23:01,23,No
2,f3ba7a96-f713-40d9-9629,152,3,12,18,10,2023-12-19,19:51:27,3,No


## Analysis
![analysis](../docs/analysis.gif)